# 03 — Summary Statistics (Person 3)

**Project question:** what drives customer satisfaction (`average_rating`) on Amazon products?

This notebook covers the Submission 1 EDA tasks:
- **4a** summary statistics for the numerical variables
- **4b** analysis of the categorical variables
- **4c** summary tables

All logic lives in `src/stats.py`; this notebook only calls those functions and interprets the results.

**Input:** the cleaned product table from Person 2 (`src/clean.py`), one row per product, 4 source categories.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

from src import stats

## Load the cleaned data

If `data/processed/products_clean.parquet` exists (built from Person 2's `clean_products`), load only the columns this analysis needs — much faster and lighter on memory than loading the text columns. Otherwise rebuild it from the raw file with Person 1's and Person 2's modules.

In [ ]:
CLEAN_PATH = "data/processed/products_clean.parquet"
COLS = ["parent_asin", "category", "main_category", "store",
        "average_rating", "rating_number", "log_rating_number",
        "price_num", "log_price", "has_price", "n_features", "n_description"]

if os.path.exists(CLEAN_PATH):
    df = pd.read_parquet(CLEAN_PATH, columns=COLS)
else:
    from src.collect import load_raw
    from src.clean import clean_products
    df = clean_products(load_raw())[COLS]

print("Rows:", f"{len(df):,}", "| Columns:", df.shape[1])
df.head()

In [ ]:
df.dtypes

## 4a — Numerical variables

`numeric_summary` gives count, missing, mean, median, spread, quartiles and **skew** for the three core numeric variables. Skew matters: a large positive skew means a few extreme products pull the mean up, so the median is the better "typical" value.

`price_num` is missing for products with no listed price, so its row describes **priced products only**.

In [ ]:
num = stats.numeric_summary(df)
num

The same summary for the log-transformed and text-length columns Person 2 created:

In [ ]:
stats.numeric_summary(df, ["log_rating_number", "log_price", "n_features", "n_description"])

## 4b — Categorical variables

Overview first: how many distinct values each categorical column has, the most common value, and how much of it is the `"Unknown"` placeholder Person 2 used for missing values.

In [ ]:
stats.categorical_overview(df)

### Source category (our 4 categories)

In [ ]:
stats.categorical_summary(df, "category")

### Amazon's own `main_category` label

In [ ]:
stats.categorical_summary(df, "main_category", top=15)

### Store — top 15 by product count
`store` has very many distinct values, so only the top rows are shown.

In [ ]:
stats.categorical_summary(df, "store", top=15)

## 4c — Summary tables

### Rating by category
Per category: product count, rating centre and spread, `diff_from_overall` (distance from the all-products mean), share of products with a price, price, and median review count.

In [ ]:
by_cat = stats.rating_by_category(df)
by_cat

### Products with vs without a listed price
About half the products have no price. Do they rate differently? If yes, anything we say about price only describes the priced half.

In [ ]:
stats.rating_by_group(df, "has_price")

### Mean rating by price quartile, per category
Quartile cut points are computed over **all priced products together**, so each band covers the same dollar range in every category.

In [ ]:
bands = stats.price_band_summary(df)
print("Cut points ($):", bands.attrs["cut_points"])
bands

Number of products behind each cell above:

In [ ]:
bands.attrs["counts"]

### Mean rating by review volume, per category
This speaks to the project goal of finding **heavily reviewed products that still rate low**. Check the counts table before trusting a small cell.

In [ ]:
vol = stats.review_volume_summary(df)
vol

In [ ]:
vol.attrs["counts"]

### Top 20 stores by product count
`"Unknown"` is excluded because it is not a real store.

In [ ]:
stores = stats.top_stores(df, n=20)
stores

Share of all products sold by these 20 stores:

In [ ]:
print(f"{100 * stores['n_products'].sum() / len(df):.2f}% of products")

### Correlations
Spearman (rank-based) by default, because `rating_number` and `price` are heavily skewed. Pearson shown below for comparison. Price correlations use priced products only.

In [ ]:
stats.correlation_table(df)

In [ ]:
stats.correlation_table(df, method="pearson")

## Preliminary insights

> **Fill these in from the tables above after running the notebook.** Every claim must quote a number from a table in this notebook and name the table. If you cannot point at the cell, cut the claim.

1. **Overall rating level** — *from `numeric_summary`:* mean vs median `average_rating`, and what the skew says about how ratings are distributed.
2. **Category differences** — *from `rating_by_category`:* which category sits highest/lowest, and by how much (`diff_from_overall`).
3. **Price** — *from `price_band_summary` and `rating_by_group(df, "has_price")`:* does mean rating rise or fall across price quartiles, and is it consistent across categories? Remember: priced products only.
4. **Review volume** — *from `review_volume_summary` and the correlation table:* do heavily reviewed products rate higher or lower? Any category where high-volume products still rate low?
5. **Stores** — *from `top_stores`:* do the biggest stores rate above or below the overall mean?

## Next steps (Submission 2)
- Test whether the category differences are statistically meaningful (ANOVA with effect size).
- Price–rating correlation with confidence intervals, per category.
- Merge review-level data to add verified-purchase share and review dates.